# 03 — Scenario builder

A **scenario** in Digicities is a named snapshot of a digital twin at a moment in time (or under a set of assumptions). The Streamlit UI's Scenario Builder lets you pick one as a *baseline* and derive modified versions from it.

To drive that engine programmatically you need two things:

1. A way to **pull a baseline out of GraphDB** into the Python dict structure the assumptions engine expects
2. A mental model of that structure so you can build scenarios from scratch too

This notebook covers both. Output of this notebook feeds directly into [`04_assumptions.ipynb`](04_assumptions.ipynb).

## 3.1 The scenario dict shape

The assumptions engine (`backend.assumptions`) operates on a plain dict:

```python
{
  "scenario_name": "baseline",
  "components": [
    {
      "uri":   "https://.../BuildingA",
      "type":  "EnergyConsumer",
      "label": "Building A (SFH with PV)",
      "attributes": {
        "Annual electricity demand": {
          "type":           "PhysicalAttribute",
          "attribute_type": "PhysicalAttribute",
          "value":          "4800.0",
          "unit":           "KiloW-HR",
          "category":       "physical",
        },
        # …
      },
      "nested_properties": {}
    },
    # …
  ],
  "component_links": []
}
```

That's it — no classes, no ORM. You can build it from a SPARQL query, from an Excel sheet, or from scratch in Python. The engines downstream only care about the shape.

## 3.2 Fetch the Alpine Village as a scenario

One SPARQL query pulls every `(component, attribute, value, unit)` tuple for the named graph; we then group them into the nested dict shape.

In [ ]:
import os, sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))
os.environ.setdefault("TRIPLESTORE_BACKEND", "fuseki")
os.environ.setdefault("GRAPHDB_URL", "http://localhost:3030")

from backend.graphdb import GraphDBClient

client = GraphDBClient(token="local", selected_repo="workspace_demo")
SAMPLE_GRAPH = "<https://digicities.info/tutorial/alpine_village>"

In [ ]:
def fetch_scenario(client, graph_uri: str, scenario_name: str) -> dict:
    """Fetch every (component, attribute, value, unit) tuple from a named graph
    and fold them into the scenario dict shape consumed by the assumptions engine.

    An *attribute* is identified by the ``dici_onto:hasAttribute`` edge that
    reaches it — never by string-matching its class name. That matters because a
    workspace's attributes are often typed with **extension** classes (here
    ``alpine:AnnualElectricityDemand``, ``alpine:FloorArea``) that don't contain
    the word "Attribute" at all. Querying the graph structure instead of the
    class name is the rule everywhere in Digicities."""
    df = client.sparql_api_query(f"""
        PREFIX dici_onto: <https://digicities.info/ontology#>
        PREFIX qudt:      <http://qudt.org/schema/qudt/>
        PREFIX rdfs:      <http://www.w3.org/2000/01/rdf-schema#>

        SELECT ?comp ?comp_label ?comp_type ?attr_label ?attr_type ?value ?unit WHERE {{
          GRAPH {graph_uri} {{
            ?comp a ?comp_type ; rdfs:label ?comp_label .
            OPTIONAL {{
              ?comp dici_onto:hasAttribute ?attr .
              ?attr rdfs:label ?attr_label .
              OPTIONAL {{ ?attr a ?attr_type }}
              OPTIONAL {{ ?attr qudt:value ?value }}
              OPTIONAL {{ ?attr dici_onto:hasUnitLabel ?unit }}
            }}
          }}
          # Keep only components: their type is a subclass of dici_onto:Component
          # in the core ontology (default graph). Attribute nodes are reached via
          # hasAttribute above, so they never show up as ?comp here.
          ?comp_type rdfs:subClassOf* dici_onto:Component .
        }}
    """, out_format="df")

    components = {}
    for _, row in df.iterrows():
        uri = row["comp"]
        comp_type = row["comp_type"].rsplit("#", 1)[-1]
        if uri not in components:
            components[uri] = {
                "uri":   uri,
                "type":  comp_type,
                "label": row["comp_label"],
                "attributes": {},
                "nested_properties": {},
            }
        if row.get("attr_label"):
            attr_type = (row["attr_type"] or "").rsplit("#", 1)[-1]
            components[uri]["attributes"][row["attr_label"]] = {
                "type":           attr_type,
                "attribute_type": attr_type,
                "value":          row.get("value") or "",
                "unit":           row.get("unit") or "",
                "category":       "physical",
            }

    return {
        "scenario_name":    scenario_name,
        "components":       list(components.values()),
        "component_links":  [],
    }

baseline = fetch_scenario(client, SAMPLE_GRAPH, "alpine_village_baseline")
print(f"Scenario '{baseline['scenario_name']}' has {len(baseline['components'])} components")

In [ ]:
# Look at Building B — the apartment block with the heat pump
from pprint import pprint
building_b = next(c for c in baseline["components"] if c["label"].startswith("Building B"))
pprint(building_b)

## 3.3 Persist the baseline

In the UI, `st.session_state.assumptions_baseline_scenario` holds this dict across reruns. Outside Streamlit, write it to disk — the next notebook reads it back.

In [ ]:
import json
pathlib.Path("sample_data").mkdir(exist_ok=True)
out = pathlib.Path("sample_data/alpine_village_baseline.json")
out.write_text(json.dumps(baseline, indent=2, default=str))
print(f"Wrote {out} ({out.stat().st_size} bytes)")

## 3.4 You can also build scenarios by hand

Nothing forces scenarios to come from GraphDB. If you're sketching a "what if we rebuild the village from scratch?" baseline, a plain literal dict works identically — the engine doesn't know or care where it came from.

In [ ]:
hand_crafted = {
    "scenario_name": "greenfield_sketch",
    "components": [
        {
            "uri":   "https://digicities.info/tutorial/greenfield/HouseX",
            "type":  "EnergyConsumer",
            "label": "Hypothetical net-zero house",
            "attributes": {
                "Annual electricity demand": {
                    "type":           "PhysicalAttribute",
                    "attribute_type": "PhysicalAttribute",
                    "value":          "3200.0",
                    "unit":           "KiloW-HR",
                    "category":       "physical",
                },
            },
            "nested_properties": {},
        }
    ],
    "component_links": [],
}
hand_crafted["components"][0]["attributes"]

## Next

[`04_assumptions.ipynb`](04_assumptions.ipynb) — take this baseline and apply single, series, and manual assumptions to generate derived scenarios.